In [ ]:
using HDF5
using PyPlot
include("PostProcess.jl")

In [ ]:
ice_shell_thickness_range = range(10.0,60.0,15)  # Define your ice shell thickness range
wavelength_range = range(5.0,300.0,14)  # Define your wavelength range
amplitude = 20.0  # Define the amplitude value
gravity = 0.113 # Define gravity
folder_name = "Enceladus"
output_path = folder_name*"_combined_output.hdf5"  # Define the output file path
combine_hdf5_files(ice_shell_thickness_range,wavelength_range,amplitude,gravity,output_path,folder_name)

In [ ]:
fname = output_path
# Displaying all of the file's information
fid = h5open(fname,"r")

# Reading Data from file
modeldata = fid["Combined Model Run"];

# Displaying the contents in Model Run group 
for obj in modeldata
    data = read(obj)
    println(obj)
    display(data)
end

# # Accessing Model Run contents that I want
Wavelength = read(modeldata,"Wavelength");
Ice_shell = read(modeldata, "Ice Shell Thickness");
Time_halfspace = read(modeldata,"Viscous Relaxation Time(Half-Space)");
Time_rel = read(modeldata,"Viscous Relaxation Time(Model)");
Time_thickening = read(modeldata, "Thickening Time");
Time_thickening_fit = read(modeldata, "Fitted Thickening Time");
Time_rel_fit = read(modeldata, "Fitted Viscous Relaxation Time");

# Close file
close(fid)

t_prime = Time_rel_fit./Time_thickening_fit;

In [ ]:
# println(Time_halfspace[1,1])
# println(Time_rel[1,1])
# println(Time_rel_fit[1,1])
# println(Time_thickening[1,1])
# println(Time_thickening_fit[1,1])

In [ ]:
cm = PyPlot.matplotlib[:cm]
LogNorm = PyPlot.matplotlib[:colors][:LogNorm]
contourf(Ice_shell/1e3,Wavelength/1e3,t_prime,norm=LogNorm(),cmap=cm.RdYlBu_r)
colorbar()

In [ ]:
# Make the plot showing the ratio of timescales (t')
cm = PyPlot.matplotlib[:cm]
LogNorm = PyPlot.matplotlib[:colors][:LogNorm]

figure()
# make the smooth-looking contour plot in the background:
levs = exp10.(range(-5,1,length=128))
# cs1=contourf(Ice_shell/1e3,(2*Wavelength)/1e3,t_prime,levs,norm=LogNorm(),cmap=cm.RdYlBu_r)
cs1=contourf(Ice_shell/1e3,(2*Wavelength)/1e3,t_prime,levs,norm=LogNorm(),vmin=1e-5,vmax=1e1,cmap=cm.RdYlBu_r)
levs1 = exp10.([-5,-4,-3,-2,-1,0,1])
# add just a selct few contour lines in the foreground
cs = contour(Ice_shell/1e3,(2*Wavelength)/1e3,t_prime,levs1,norm=LogNorm(),colors="k")
clabel(cs,inline=true,fmt="%2.e",colors="k",fontsize=12)
title("Enceladus",fontsize=14)
colorbar(cs1,ticks=levs1,label=L"\log_{10}(t')")
gca().yaxis.set_label_position("left")
gca().set_ylabel(L"\lambda"*" (km)",fontsize=14)
gca().set_xlabel("H (km)",fontsize=14,rotation=0)

# Highlight region with a square
x0,y0 = 10,600  # replace with the coordinates of the bottom-left corner
width,height = 8,-100  # replace with the dimensions of the square
# Making square
gca().add_patch(PyPlot.matplotlib[:patches][:Rectangle](
        (x0, y0),
        width,
        height,
        fill=true,
        facecolor="gray",
        alpha=0.7,
        linewidth=5))

# Making texts
annotate("SPT-like", 
        xy=(x0,y0), 
        xytext=(x0+8,y0-50), 
        fontsize=16, 
        color="gray")

annotate("(a)", 
        xy=(x0,y0), 
        xytext=(x0-10,y0), 
        fontsize=14, 
        color="black")

tight_layout()
# savefig("/Users/christianaguirre/tmp/Enceladus_tprime.pdf",dpi=300)
# savefig("/Users/christianaguirre/MS_Thesis/Enceladus_tprime.pdf",dpi=300)
show()

In [ ]:
contourf(Ice_shell/1e3,(2*Wavelength)/1e3,Time_rel_fit,norm=LogNorm())
colorbar()

In [ ]:
figure()
# smooth background contour plot:
levs = exp10.(range(2, stop=7, length=100))
cs = contourf(Ice_shell/1e3,(2*Wavelength)/1e3,Time_rel_fit,levs,norm=LogNorm(),vmin=1e2,vmax=1e7)
# contour levels to highlight
levs1 = exp10.([2,3,4,5,6,7])
cs1 = contour(Ice_shell/1e3,(2*Wavelength)/1e3,Time_rel_fit,levs1,norm=LogNorm(),colors="k")

# use the scientific notation formatter to make a dictionary of contour labels:
fmt = PyPlot.matplotlib[:ticker][:LogFormatterSciNotation]()
labels = Dict()
for level in cs1.levels
    labels[level] = fmt(level)
end
clabel(cs1,inline=true,fmt=labels,colors="k",fontsize=12)
colorbar(cs,ticks=levs1,label="Relaxation Timescale (yr)")
title(L"t_{relaxation}",fontsize=14)
gca().yaxis.set_label_position("left")
gca().set_ylabel(L"\lambda"*" (km)",fontsize=14)
gca().set_xlabel("H (km)",fontsize=14,rotation=0)

# Making text
annotate("(a)", 
        xy=(x0,y0), 
        xytext=(x0-10,y0), 
        fontsize=14, 
        color="black")

tight_layout()
# savefig("/Users/christianaguirre/tmp/Enceladus_trel.pdf",dpi=300)
# savefig("/Users/christianaguirre/MS_Thesis/Enceladus_trel.pdf",dpi=300)
show()

In [ ]:
ticker = PyPlot.matplotlib[:ticker]
figure()
cs = contour(Ice_shell/1e3,Wavelength/1e3,Time_thickening_fit,colors="k")
contourf(cs,cmap=get_cmap("viridis"))
title(L"t_{thickening}",fontsize=14)
clabel(cs,inline=true,fmt="%1.f",colors="k",fontsize=12)
cbar = colorbar(label="Thickening Timescale (yr)",location="right")
formatter = ticker.ScalarFormatter(useMathText=true)
formatter.set_scientific(true)
formatter.set_powerlimits((-2, 2))
formatter.set_useOffset(false)
cbar.ax.yaxis.set_major_formatter(formatter)
gca().yaxis.set_label_position("left")
gca().set_ylabel(L"\lambda"*" (km)",fontsize=14)
gca().set_xlabel("H (km)",fontsize=14,rotation=0)
tight_layout()
show()

In [ ]:
using Statistics
figure()
# Define the 3 levels to highlight
highlight_levels = [0.5e7,1.0e7,1.5e7,2.0e7]
# Plot all 32 levels with muted lines
cs = contour(Ice_shell/1e3, (2*Wavelength)/1e3, Time_thickening_fit,levels=100)
# Plot only the 3 highlight levels with a thicker line or different color
highlight_cs = contour(Ice_shell/1e3,(2*Wavelength)/1e3,Time_thickening_fit, colors="k",levels=highlight_levels)
# Fill the contours with the colormap
contourf(cs,cmap=get_cmap("viridis"))
# Add title and contour labels
title(L"t_{thickening}",fontsize=14)
clabel(highlight_cs,inline=true,fmt="%1.0f",colors="k",fontsize=12)  # Labels for highlighted levels
# Add the colorbar
cbar = colorbar(label="Thickening Timescale (yr)", location="right")
# Set the ScalarFormatter for the colorbar to use scientific notation
formatter = ticker.ScalarFormatter(useMathText=true)
formatter.set_scientific(true)
formatter.set_powerlimits((-2, 2))
formatter.set_useOffset(false)
cbar.ax.yaxis.set_major_formatter(formatter)
# Set labels and formatting
gca().yaxis.set_label_position("left")
gca().set_ylabel(L"\lambda"*" (km)", fontsize=14)
gca().set_xlabel("H (km)", fontsize=14, rotation=0)

# Making text
annotate("(b)", 
        xy=(x0,y0), 
        xytext=(x0-10,y0), 
        fontsize=14, 
        color="black")

# Adjust layout and save the figures
tight_layout()
# savefig("/Users/christianaguirre/tmp/Enceladus_ttic.pdf",dpi=300)
# savefig("/Users/christianaguirre/MS_Thesis/Enceladus_ttic.pdf",dpi=300)
show()

In [ ]:
# fig = figure(figsize=(22,6))
# # gca().set_ylabel("depth (km)",fontsize=12,labelpad=25)
# gca().set_ylabel(L"\lambda"*" (km)",fontsize=14)
# gca().yaxis.set_label_coords(-0.03,0.5)
# # Remove axis lines
# gca().spines["top"].set_visible(false)
# gca().spines["right"].set_visible(false)
# gca().spines["left"].set_visible(false)
# gca().spines["bottom"].set_visible(false)
# # Remove ticks
# gca().xaxis.set_ticks([])
# gca().yaxis.set_ticks([])
# fig.subplots_adjust(hspace=0.15)
# cm = PyPlot.matplotlib[:cm]
# LogNorm = PyPlot.matplotlib[:colors][:LogNorm]

# subplot(131)
# ax1 = gca()
# levsf = exp10.(range(-3,2,length=128))
# cs1=contourf(Ice_shell/1e3,Wavelength/1e3,t_prime1,levsf,norm=LogNorm(),vmin=1e-6,vmax=1e1,cmap=cm.RdYlBu_r)
# levs1 = exp10.([-3,-2,-1,0,1,2,3])
# # add just a selct few contour lines in the foreground
# cs = contour(Ice_shell/1e3,Wavelength/1e3,t_prime1,levs1,norm=LogNorm(),colors="k")
# clabel(cs,inline=true,colors="k",fontsize=10)
# title(strip(L"\mu\,=\,10^{13}"))
# colorbar(cs1,ticks=levs1,label=L"\log_{10}(t')")
# ax1.set_xlabel(L"H"*" (km)",fontsize=14,rotation=0)

# subplot(132)
# ax3 = gca()
# levst = exp10.(range(-3,3,length=128))
# # cs1=contourf(Ice_shell/1e3,Wavelength/1e3,t_prime,levs,norm=LogNorm(),cmap=cm.RdYlBu_r)
# cs1=contourf(Ice_shell/1e3,Wavelength/1e3,t_prime3,levst,norm=LogNorm(),vmin=1e-2,vmax=1e2,cmap=cm.RdYlBu_r)
# levs3= exp10.([-3,-2,-1,0,1,2,3])
# # add just a selct few contour lines in the foreground
# cs = contour(Ice_shell/1e3,Wavelength/1e3,t_prime3,levs3,norm=LogNorm(),colors="k")
# clabel(cs,inline=true,colors="k",fontsize=10)
# title(strip(L"\mu\,=\,10^{14}"))
# colorbar(cs1,ticks=levs1,label=L"\log_{10}(t')")
# ax3.set_xlabel(L"H"*" (km)",fontsize=14,rotation=0)
# ax3.get_yaxis().set_visible(false)

# subplot(133)
# ax2 = gca()
# levss = exp10.(range(-2,4,length=128))
# # cs1=contourf(Ice_shell/1e3,Wavelength/1e3,t_prime,levs,norm=LogNorm(),cmap=cm.RdYlBu_r)
# cs1=contourf(Ice_shell/1e3,Wavelength/1e3,t_prime2,levss,norm=LogNorm(),vmin=1e-2,vmax=1e2,cmap=cm.RdYlBu_r)
# levs2 = exp10.([-2,-1,0,1,2,3])
# # add just a selct few contour lines in the foreground
# cs = contour(Ice_shell/1e3,Wavelength/1e3,t_prime2,levs2,norm=LogNorm(),colors="k")
# clabel(cs,inline=true,colors="k",fontsize=10)
# title(strip(L"\mu\,=\,10^{15}"))
# colorbar(cs1,ticks=levs1,label=L"\log_{10}(t')")
# ax2.set_xlabel(L"H"*" (km)",fontsize=14,rotation=0)
# ax2.get_yaxis().set_visible(false)

# show()